# Notebook 03 — Patient-Level Representation

## Patient Similarity Network + Agent-Based Modelling for Readmission

### Purpose
Aggregate the cleaned encounter-level Diabetes 130-US Hospitals data by `patient_nbr`
and create **one reproducible row per patient**.

This notebook follows the project proposal:

- encounter counts
- inpatient / emergency / outpatient utilization
- medication burden
- diagnosis burden
- procedure burden
- demographic/context variables
- a separate readmission target summary

### Important methodological rule
The dataset does **not** contain encounter dates/timestamps. Therefore, this notebook
does **not** invent a temporal ordering and does not call cumulative utilization "prior
utilization". Notebook 04 will define the prediction/intervention point and perform the
leakage-safe target and split work.

The readmission outcome is preserved separately and is **never used to construct the
patient feature representation**.

## Expected project structure

```text
sna/
├── diabetes+130-us+hospitals+for+years+1999-2008/
│   ├── diabetic_data.csv
│   └── IDS_mapping.csv
├── notebooks/
│   ├── 01_dataset_audit.ipynb
│   ├── 02_cleaning.ipynb
│   └── 03_patient_representation.ipynb
├── results/
└── figures/
```

Notebook 03 reads the output created by Notebook 02:

`results/02_cleaned_encounters.csv`

and writes all new outputs to `results/`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

SEED = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Robust project-root discovery:
# this works when the notebook is opened from the notebooks/ directory
# or when VS Code uses the project root as the working directory.
cwd = Path.cwd()

candidate_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / "results" / "02_cleaned_encounters.csv").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find results/02_cleaned_encounters.csv. "
        "Open this notebook inside the sna project folder."
    )

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PATH = RESULTS_DIR / "02_cleaned_encounters.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT:", CLEAN_PATH)

PROJECT_ROOT: c:\Users\Gayatri\OneDrive\Desktop\sna
INPUT: c:\Users\Gayatri\OneDrive\Desktop\sna\results\02_cleaned_encounters.csv


In [2]:
# Load cleaned encounter data produced by Notebook 02
df = pd.read_csv(CLEAN_PATH, low_memory=False)

print("Cleaned encounter table shape:", df.shape)
print("Unique patients:", df["patient_nbr"].nunique())
print("Unique encounters:", df["encounter_id"].nunique())

assert "patient_nbr" in df.columns
assert "encounter_id" in df.columns
assert "readmitted" in df.columns
assert df["encounter_id"].is_unique, "Encounter IDs must remain unique."

print("\nInput columns:")
print(df.columns.tolist())

Cleaned encounter table shape: (101766, 50)
Unique patients: 71518
Unique encounters: 101766

Input columns:
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


## 1. Freeze variables that cannot be patient-level features

The following are excluded from the feature representation:

- `patient_nbr` — identifier
- `encounter_id` — identifier
- `readmitted` — target/outcome
- `discharge_disposition_id` — retained for later temporal review
- `time_in_hospital` — retained for later temporal review

Variables with at least 90% missingness are also excluded from the representation.

This is deliberately conservative. Notebook 04 will make the final leakage-safe feature
eligibility decision after the prediction/intervention point is defined.

In [3]:
IDENTIFIER_COLS = ["encounter_id", "patient_nbr"]
TARGET_COL = "readmitted"
REVIEW_REQUIRED_COLS = ["discharge_disposition_id", "time_in_hospital"]

missing_pct = df.isna().mean()

HIGH_MISSING_COLS = [
    c for c in df.columns
    if c not in IDENTIFIER_COLS + [TARGET_COL]
    and missing_pct[c] >= 0.90
]

EXCLUDED_FROM_REPRESENTATION = sorted(set(
    IDENTIFIER_COLS
    + [TARGET_COL]
    + REVIEW_REQUIRED_COLS
    + HIGH_MISSING_COLS
))

print("High-missingness exclusions (>=90%):", HIGH_MISSING_COLS)
print("Review-required exclusions:", REVIEW_REQUIRED_COLS)
print("Total excluded from current representation:", len(EXCLUDED_FROM_REPRESENTATION))

High-missingness exclusions (>=90%): ['weight', 'max_glu_serum']
Review-required exclusions: ['discharge_disposition_id', 'time_in_hospital']
Total excluded from current representation: 7


## 2. Numeric patient-level representation

For encounter-level numeric variables, we create clinically interpretable summaries:

- `encounter_count`
- total / mean / maximum for utilization variables
- total / mean / maximum for laboratory, procedure, medication, and diagnosis counts

The resulting variables describe the patient's **observed records in the available dataset**.
They are not treated as temporally prior variables yet.

In [4]:
# Core utilization variables required by the proposal
UTILIZATION_COLS = [
    c for c in [
        "number_inpatient",
        "number_emergency",
        "number_outpatient",
    ] if c in df.columns
]

COUNT_SUMMARY_COLS = [
    c for c in [
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_diagnoses",
    ]
    if c in df.columns
]

# Convert known numeric candidates safely.
for c in UTILIZATION_COLS + COUNT_SUMMARY_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Utilization columns:", UTILIZATION_COLS)
print("Clinical count columns:", COUNT_SUMMARY_COLS)

Utilization columns: ['number_inpatient', 'number_emergency', 'number_outpatient']
Clinical count columns: ['num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses']


In [5]:
# Base aggregation: exactly one group per patient
patient_base = (
    df.groupby("patient_nbr", sort=True)
      .size()
      .rename("encounter_count")
      .to_frame()
)

# Aggregate requested utilization and clinical count variables.
for c in UTILIZATION_COLS + COUNT_SUMMARY_COLS:
    g = df.groupby("patient_nbr", sort=True)[c]
    patient_base[f"{c}_sum"] = g.sum(min_count=1)
    patient_base[f"{c}_mean"] = g.mean()
    patient_base[f"{c}_max"] = g.max()

patient_base.head()

,encounter_count,number_inpatient_sum,number_inpatient_mean,number_inpatient_max,number_emergency_sum,number_emergency_mean,number_emergency_max,number_outpatient_sum,number_outpatient_mean,number_outpatient_max,num_lab_procedures_sum,num_lab_procedures_mean,num_lab_procedures_max,num_procedures_sum,num_procedures_mean,num_procedures_max,num_medications_sum,num_medications_mean,num_medications_max,number_diagnoses_sum,number_diagnoses_mean,number_diagnoses_max
patient_nbr,,,,,,,,,,,,,,,,,,,,,,
135,2,1,0.5,1,0,0.0,0,0,0.0,0,108,54.0,77,7,3.5,6,47,23.5,33,13,6.5,8
378,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.0,49,1,1.0,1,11,11.0,11,3,3.0,3
729,1,0,0.0,0,0,0.0,0,0,0.0,0,68,68.0,68,2,2.0,2,23,23.0,23,9,9.0,9
774,1,0,0.0,0,0,0.0,0,0,0.0,0,46,46.0,46,0,0.0,0,20,20.0,20,9,9.0,9
927,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.0,49,0,0.0,0,5,5.0,5,3,3.0,3


## 3. Medication burden

The Diabetes 130-US Hospitals dataset contains medication-status variables such as
`metformin`, `insulin`, and other diabetes medications.

For each encounter, a medication is considered **present in the recorded medication
profile** when its value is not missing and is not `"No"`.

We then calculate per patient:

- total medication-status entries recorded
- number of distinct medication variables ever recorded as active/non-`No`
- average number of active medication variables per encounter

These are representation variables only; they do not use `readmitted`.

In [6]:
# Medication-status columns: all columns between the clinical diagnosis/procedure area
# and the final outcome are detected using the standard Diabetes dataset naming pattern.
MEDICATION_COLS = [
    c for c in df.columns
    if c.lower() in {
        "metformin", "repaglinide", "nateglinide", "chlorpropamide",
        "glimepiride", "acetohexamide", "glipizide", "glyburide",
        "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
        "miglitol", "troglitazone", "tolazamide", "examide",
        "citoglipton", "insulin", "glyburide-metformin",
        "glipizide-metformin", "glimepiride-pioglitazone",
        "metformin-rosiglitazone", "metformin-pioglitazone"
    }
]

print("Medication columns detected:", len(MEDICATION_COLS))
print(MEDICATION_COLS)

Medication columns detected: 23
['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


In [7]:
if MEDICATION_COLS:
    med = df[MEDICATION_COLS].copy()
    med = med.astype("string")
    med_present = med.notna() & med.ne("No")

    df["_med_active_count"] = med_present.sum(axis=1)

    med_by_patient = df.groupby("patient_nbr", sort=True)["_med_active_count"].agg(
        medication_active_entries_sum="sum",
        medication_active_entries_mean="mean",
        medication_active_entries_max="max"
    )

    # Number of distinct medication variables with at least one non-No observation
    med_distinct = med_present.groupby(df["patient_nbr"]).sum().gt(0).sum(axis=1)
    med_distinct.name = "distinct_medications_recorded_active"

    patient_base = patient_base.join(med_by_patient).join(med_distinct)
else:
    patient_base["medication_active_entries_sum"] = 0
    patient_base["medication_active_entries_mean"] = 0
    patient_base["medication_active_entries_max"] = 0
    patient_base["distinct_medications_recorded_active"] = 0

patient_base.head()

,encounter_count,number_inpatient_sum,number_inpatient_mean,number_inpatient_max,number_emergency_sum,number_emergency_mean,number_emergency_max,number_outpatient_sum,number_outpatient_mean,number_outpatient_max,num_lab_procedures_sum,num_lab_procedures_mean,num_lab_procedures_max,num_procedures_sum,num_procedures_mean,num_procedures_max,num_medications_sum,num_medications_mean,num_medications_max,number_diagnoses_sum,number_diagnoses_mean,number_diagnoses_max,medication_active_entries_sum,medication_active_entries_mean,medication_active_entries_max,distinct_medications_recorded_active
patient_nbr,,,,,,,,,,,,,,,,,,,,,,,,,,
135,2,1,0.5,1,0,0.0,0,0,0.0,0,108,54.0,77,7,3.5,6,47,23.5,33,13,6.5,8,5,2.5,3,3
378,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.0,49,1,1.0,1,11,11.0,11,3,3.0,3,0,0.0,0,0
729,1,0,0.0,0,0,0.0,0,0,0.0,0,68,68.0,68,2,2.0,2,23,23.0,23,9,9.0,9,1,1.0,1,1
774,1,0,0.0,0,0,0.0,0,0,0.0,0,46,46.0,46,0,0.0,0,20,20.0,20,9,9.0,9,2,2.0,2,2
927,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.0,49,0,0.0,0,5,5.0,5,3,3.0,3,1,1.0,1,1


## 4. Diagnosis burden

The dataset provides three diagnosis-code fields:

- `diag_1`
- `diag_2`
- `diag_3`

We calculate:

- number of diagnosis-code entries observed
- number of unique diagnosis codes observed across all encounters
- number of encounters with at least one diagnosis

The actual diagnosis codes are retained separately as optional categorical information,
but the main representation uses burden/count variables so that Notebook 05 can perform
controlled encoding and similarity construction.

In [8]:
DIAG_COLS = [c for c in ["diag_1", "diag_2", "diag_3"] if c in df.columns]

if DIAG_COLS:
    diag_present = df[DIAG_COLS].notna()
    df["_diagnosis_entry_count"] = diag_present.sum(axis=1)

    diag_entries = (
        df[["patient_nbr"] + DIAG_COLS]
        .melt(id_vars="patient_nbr", value_name="diagnosis_code")
        .dropna(subset=["diagnosis_code"])
    )
    diag_entries["diagnosis_code"] = diag_entries["diagnosis_code"].astype(str).str.strip()
    diag_entries = diag_entries[diag_entries["diagnosis_code"] != ""]

    unique_diag = (
        diag_entries.groupby("patient_nbr")["diagnosis_code"]
        .nunique()
        .rename("unique_diagnosis_codes")
    )

    diag_by_patient = df.groupby("patient_nbr")["_diagnosis_entry_count"].agg(
        diagnosis_entries_sum="sum",
        diagnosis_entries_mean="mean",
        diagnosis_entries_max="max"
    )

    patient_base = patient_base.join(diag_by_patient).join(unique_diag)
else:
    patient_base["diagnosis_entries_sum"] = 0
    patient_base["diagnosis_entries_mean"] = 0
    patient_base["diagnosis_entries_max"] = 0
    patient_base["unique_diagnosis_codes"] = 0

patient_base = patient_base.fillna({
    "unique_diagnosis_codes": 0
})

patient_base.head()

,encounter_count,number_inpatient_sum,number_inpatient_mean,number_inpatient_max,number_emergency_sum,number_emergency_mean,number_emergency_max,number_outpatient_sum,number_outpatient_mean,number_outpatient_max,num_lab_procedures_sum,num_lab_procedures_mean,num_lab_procedures_max,num_procedures_sum,num_procedures_mean,num_procedures_max,num_medications_sum,num_medications_mean,num_medications_max,number_diagnoses_sum,number_diagnoses_mean,number_diagnoses_max,medication_active_entries_sum,medication_active_entries_mean,medication_active_entries_max,distinct_medications_recorded_active,diagnosis_entries_sum,diagnosis_entries_mean,diagnosis_entries_max,unique_diagnosis_codes
patient_nbr,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
135,2,1,0.5,1,0,0.0,0,0,0.0,0,108,54.0,77,7,3.5,6,47,23.5,33,13,6.5,8,5,2.5,3,3,6,3.0,3,6.0
378,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.0,49,1,1.0,1,11,11.0,11,3,3.0,3,0,0.0,0,0,3,3.0,3,3.0
729,1,0,0.0,0,0,0.0,0,0,0.0,0,68,68.0,68,2,2.0,2,23,23.0,23,9,9.0,9,1,1.0,1,1,3,3.0,3,3.0
774,1,0,0.0,0,0,0.0,0,0,0.0,0,46,46.0,46,0,0.0,0,20,20.0,20,9,9.0,9,2,2.0,2,2,3,3.0,3,3.0
927,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.0,49,0,0.0,0,5,5.0,5,3,3.0,3,1,1.0,1,1,3,3.0,3,3.0


## 5. Procedure burden

The dataset does not provide a detailed procedure-code field. It provides
`num_procedures`, which is already included in the numeric summary above.

Therefore, we do **not** invent a unique-procedure-code measure.

The procedure representation is:

- total procedures across recorded encounters
- mean procedures per encounter
- maximum procedures in an encounter

## 6. Demographic and clinical context variables

For categorical/context variables without timestamps, we use a deterministic patient-level
mode. If multiple values tie, the lexicographically smallest value is selected.

This avoids inventing an encounter order.

Variables considered include:

- `race`
- `gender`
- `age`
- `admission_type_id`
- `admission_source_id`
- `max_glu_serum`
- `A1Cresult`
- `change`
- `diabetesMed`

These are descriptive patient-level summaries. Notebook 04 will determine which variables
are admissible for leakage-safe prediction/network construction.

In [9]:
CONTEXT_MODE_COLS = [
    c for c in [
        "race",
        "gender",
        "age",
        "admission_type_id",
        "admission_source_id",
        "max_glu_serum",
        "A1Cresult",
        "change",
        "diabetesMed",
    ]
    if c in df.columns and c not in EXCLUDED_FROM_REPRESENTATION
]

def deterministic_mode(series):
    s = series.dropna().astype(str).str.strip()
    s = s[s.ne("")]
    if s.empty:
        return pd.NA
    counts = s.value_counts()
    max_count = counts.max()
    tied = sorted(counts[counts == max_count].index.tolist())
    return tied[0]

context_data = pd.DataFrame(index=patient_base.index)

for c in CONTEXT_MODE_COLS:
    context_data[f"{c}_mode"] = (
        df.groupby("patient_nbr", sort=True)[c]
          .agg(deterministic_mode)
    )

patient_base = patient_base.join(context_data)

print("Context variables:", CONTEXT_MODE_COLS)
patient_base.head()

Context variables: ['race', 'gender', 'age', 'admission_type_id', 'admission_source_id', 'A1Cresult', 'change', 'diabetesMed']


,encounter_count,number_inpatient_sum,number_inpatient_mean,number_inpatient_max,number_emergency_sum,number_emergency_mean,number_emergency_max,number_outpatient_sum,number_outpatient_mean,number_outpatient_max,num_lab_procedures_sum,num_lab_procedures_mean,num_lab_procedures_max,num_procedures_sum,num_procedures_mean,num_procedures_max,num_medications_sum,num_medications_mean,num_medications_max,number_diagnoses_sum,number_diagnoses_mean,number_diagnoses_max,medication_active_entries_sum,medication_active_entries_mean,medication_active_entries_max,distinct_medications_recorded_active,diagnosis_entries_sum,diagnosis_entries_mean,diagnosis_entries_max,unique_diagnosis_codes,race_mode,gender_mode,age_mode,admission_type_id_mode,admission_source_id_mode,A1Cresult_mode,change_mode,diabetesMed_mode
patient_nbr,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
135,2,1,0.5,1,0,0.0,0,0,0.0,0,108,54.0,77,7,3.5,6,47,23.5,33,13,6.5,8,5,2.5,3,3,6,3.0,3,6.0,Caucasian,Female,[50-60),1,1,NaN,Ch,Yes
378,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.0,49,1,1.0,1,11,11.0,11,3,3.0,3,0,0.0,0,0,3,3.0,3,3.0,Caucasian,Female,[50-60),3,1,NaN,No,No
729,1,0,0.0,0,0,0.0,0,0,0.0,0,68,68.0,68,2,2.0,2,23,23.0,23,9,9.0,9,1,1.0,1,1,3,3.0,3,3.0,Caucasian,Female,[80-90),1,7,>7,No,Yes
774,1,0,0.0,0,0,0.0,0,0,0.0,0,46,46.0,46,0,0.0,0,20,20.0,20,9,9.0,9,2,2.0,2,2,3,3.0,3,3.0,Caucasian,Female,[80-90),1,7,>8,Ch,Yes
927,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.0,49,0,0.0,0,5,5.0,5,3,3.0,3,1,1.0,1,1,3,3.0,3,3.0,AfricanAmerican,Female,[30-40),1,7,NaN,No,Yes


## 7. Assemble the final patient-level representation

`patient_nbr` remains the node/identifier column.

**Readmission is deliberately NOT included here.**

The separate target table below contains outcome summaries only for later Notebook 04
target definition and validation.

In [10]:
patient_representation = patient_base.reset_index()

# Explicit safety checks: target and encounter ID must never enter feature columns.
assert "readmitted" not in patient_representation.columns
assert "encounter_id" not in patient_representation.columns

# No duplicate patient representation.
assert not patient_representation["patient_nbr"].duplicated().any()

print("Patient representation shape:", patient_representation.shape)
print("Unique patient IDs:", patient_representation["patient_nbr"].nunique())
print("Duplicate patient IDs:", patient_representation["patient_nbr"].duplicated().sum())

patient_representation.head(10)

Patient representation shape: (71518, 39)
Unique patient IDs: 71518
Duplicate patient IDs: 0


,patient_nbr,encounter_count,number_inpatient_sum,number_inpatient_mean,number_inpatient_max,number_emergency_sum,number_emergency_mean,number_emergency_max,number_outpatient_sum,number_outpatient_mean,number_outpatient_max,num_lab_procedures_sum,num_lab_procedures_mean,num_lab_procedures_max,num_procedures_sum,num_procedures_mean,num_procedures_max,num_medications_sum,num_medications_mean,num_medications_max,number_diagnoses_sum,number_diagnoses_mean,number_diagnoses_max,medication_active_entries_sum,medication_active_entries_mean,medication_active_entries_max,distinct_medications_recorded_active,diagnosis_entries_sum,diagnosis_entries_mean,diagnosis_entries_max,unique_diagnosis_codes,race_mode,gender_mode,age_mode,admission_type_id_mode,admission_source_id_mode,A1Cresult_mode,change_mode,diabetesMed_mode
0,135,2,1,0.5,1,0,0.0,0,0,0.0,0,108,54.000000,77,7,3.500000,6,47,23.5,33,13,6.500000,8,5,2.5,3,3,6,3.0,3,6.0,Caucasian,Female,[50-60),1,1,NaN,Ch,Yes
1,378,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.000000,49,1,1.000000,1,11,11.0,11,3,3.000000,3,0,0.0,0,0,3,3.0,3,3.0,Caucasian,Female,[50-60),3,1,NaN,No,No
2,729,1,0,0.0,0,0,0.0,0,0,0.0,0,68,68.000000,68,2,2.000000,2,23,23.0,23,9,9.000000,9,1,1.0,1,1,3,3.0,3,3.0,Caucasian,Female,[80-90),1,7,>7,No,Yes
3,774,1,0,0.0,0,0,0.0,0,0,0.0,0,46,46.000000,46,0,0.000000,0,20,20.0,20,9,9.000000,9,2,2.0,2,2,3,3.0,3,3.0,Caucasian,Female,[80-90),1,7,>8,Ch,Yes
4,927,1,0,0.0,0,0,0.0,0,0,0.0,0,49,49.000000,49,0,0.000000,0,5,5.0,5,3,3.000000,3,1,1.0,1,1,3,3.0,3,3.0,AfricanAmerican,Female,[30-40),1,7,NaN,No,Yes
5,1152,5,7,1.4,2,0,0.0,0,0,0.0,0,209,41.800000,54,10,2.000000,4,81,16.2,19,24,4.800000,9,5,1.0,1,1,13,2.6,3,8.0,AfricanAmerican,Female,[60-70),1,7,NaN,No,Yes
6,1305,1,0,0.0,0,0,0.0,0,0,0.0,0,52,52.000000,52,1,1.000000,1,16,16.0,16,9,9.000000,9,1,1.0,1,1,3,3.0,3,3.0,Caucasian,Female,[60-70),3,1,NaN,No,Yes
7,1314,3,3,1.0,2,0,0.0,0,0,0.0,0,151,50.333333,54,13,4.333333,5,39,13.0,14,23,7.666667,9,4,1.333333,2,2,9,3.0,3,7.0,Caucasian,Female,[40-50),1,7,NaN,No,Yes
8,1629,1,2,2.0,2,1,1.0,1,3,3.0,3,21,21.000000,21,0,0.000000,0,15,15.0,15,7,7.000000,7,1,1.0,1,1,3,3.0,3,3.0,Caucasian,Male,[70-80),3,4,NaN,No,Yes
9,2025,1,0,0.0,0,0,0.0,0,0,0.0,0,47,47.000000,47,2,2.000000,2,18,18.0,18,9,9.000000,9,2,2.0,2,2,3,3.0,3,3.0,Caucasian,Female,[70-80),2,1,Norm,Ch,Yes


## 8. Preserve readmission outcome separately

For this notebook, we preserve the encounter-level readmission labels in a separate
patient-level summary.

This is **not** used as a feature.

The summary contains:

- number of encounters labelled `NO`
- number labelled `>30`
- number labelled `<30`
- whether the patient has any observed `<30` readmission
- whether the patient has any observed readmission (`<30` or `>30`)

Notebook 04 will define the project's exact target. Until then, these columns must not
be used for similarity, network construction, or model training.

In [11]:
TARGET_LABELS = {"NO", ">30", "<30"}

target_clean = df["readmitted"].astype("string").str.strip()

unknown_target = sorted(set(target_clean.dropna().unique()) - TARGET_LABELS)
print("Unexpected target labels:", unknown_target)
assert not unknown_target, "Unexpected readmission category found."

target_summary = (
    df.assign(readmitted_clean=target_clean)
      .groupby("patient_nbr", sort=True)["readmitted_clean"]
      .value_counts()
      .unstack(fill_value=0)
)

for label in ["NO", ">30", "<30"]:
    if label not in target_summary.columns:
        target_summary[label] = 0

target_summary = target_summary[["NO", ">30", "<30"]].copy()
target_summary.columns = [
    "encounters_no_readmission",
    "encounters_readmission_over_30d",
    "encounters_readmission_under_30d",
]

target_summary["any_observed_readmission_under_30d"] = (
    target_summary["encounters_readmission_under_30d"] > 0
).astype(int)

target_summary["any_observed_readmission"] = (
    (target_summary["encounters_readmission_under_30d"] > 0)
    | (target_summary["encounters_readmission_over_30d"] > 0)
).astype(int)

target_summary = target_summary.reset_index()

print(target_summary.head())

Unexpected target labels: []
   patient_nbr  encounters_no_readmission  encounters_readmission_over_30d  \
0          135                          0                                1   
1          378                          1                                0   
2          729                          1                                0   
3          774                          1                                0   
4          927                          1                                0   

   encounters_readmission_under_30d  any_observed_readmission_under_30d  \
0                                 1                                   1   
1                                 0                                   0   
2                                 0                                   0   
3                                 0                                   0   
4                                 0                                   0   

   any_observed_readmission  
0                    

## 9. Quality checks on the patient representation

We now verify:

1. exactly one row per patient
2. patient count matches Notebook 02
3. no duplicate patient IDs
4. no identifier leakage
5. no target leakage
6. numeric representation fields are finite
7. count/utilization variables are non-negative
8. target is separate

In [12]:
expected_patient_count = df["patient_nbr"].nunique()

numeric_representation_cols = patient_representation.select_dtypes(
    include=[np.number]
).columns.tolist()

numeric_feature_cols = [
    c for c in numeric_representation_cols
    if c != "patient_nbr"
]

finite_ok = np.isfinite(
    patient_representation[numeric_feature_cols].to_numpy(dtype=float)
).all()

count_like_cols = [
    c for c in numeric_feature_cols
    if any(token in c.lower() for token in [
        "count", "sum", "number_", "num_", "encounters_", "procedures",
        "medication", "diagnosis", "utilization"
    ])
]

nonnegative_ok = True
negative_columns = {}

for c in count_like_cols:
    vals = pd.to_numeric(patient_representation[c], errors="coerce")
    if vals.notna().any() and (vals < 0).any():
        nonnegative_ok = False
        negative_columns[c] = int((vals < 0).sum())

checks = {
    "patient_row_count_matches_nunique": len(patient_representation) == expected_patient_count,
    "patient_ids_unique": not patient_representation["patient_nbr"].duplicated().any(),
    "patient_id_present": "patient_nbr" in patient_representation.columns,
    "encounter_id_absent": "encounter_id" not in patient_representation.columns,
    "readmission_target_absent_from_features": "readmitted" not in patient_representation.columns,
    "numeric_values_finite": finite_ok,
    "count_variables_nonnegative": nonnegative_ok,
    "target_separate": "readmitted" not in patient_representation.columns,
    "target_patient_count_matches": len(target_summary) == expected_patient_count,
}

for k, v in checks.items():
    print(f"{'PASS' if v else 'FAIL'} | {k}")

if negative_columns:
    print("\nNegative count-like values found:")
    print(negative_columns)

assert all(checks.values()), "At least one patient-level representation check failed."

PASS | patient_row_count_matches_nunique
PASS | patient_ids_unique
PASS | patient_id_present
PASS | encounter_id_absent
PASS | readmission_target_absent_from_features
PASS | numeric_values_finite
PASS | count_variables_nonnegative
PASS | target_separate
PASS | target_patient_count_matches


## 10. Distribution inspection

The proposal specifically requires inspecting distributions before moving forward.

The following output is a compact diagnostic summary, not a modelling result.

In [13]:
distribution_cols = [
    c for c in [
        "encounter_count",
        "number_inpatient_sum",
        "number_emergency_sum",
        "number_outpatient_sum",
        "num_lab_procedures_mean",
        "num_procedures_mean",
        "num_medications_mean",
        "number_diagnoses_mean",
        "medication_active_entries_mean",
        "distinct_medications_recorded_active",
        "diagnosis_entries_mean",
        "unique_diagnosis_codes",
    ]
    if c in patient_representation.columns
]

distribution_summary = (
    patient_representation[distribution_cols]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.99])
    .T
)

distribution_summary

,count,mean,std,min,25%,50%,75%,90%,99%,max
encounter_count,71518.0,1.422942,1.09074,1.0,1.0,1.0,1.0,2.0,6.0,40.0
number_inpatient_sum,71518.0,0.904374,4.165278,0.0,0.0,0.0,1.0,2.0,13.0,382.0
number_emergency_sum,71518.0,0.28151,2.708646,0.0,0.0,0.0,0.0,0.0,5.0,313.0
number_outpatient_sum,71518.0,0.525574,2.356121,0.0,0.0,0.0,0.0,1.0,9.0,149.0
num_lab_procedures_mean,71518.0,43.22311,18.893236,1.0,32.0,44.0,56.0,67.0,84.0,121.0
num_procedures_mean,71518.0,1.412397,1.663913,0.0,0.0,1.0,2.0,4.0,6.0,6.0
num_medications_mean,71518.0,15.786512,7.913624,1.0,10.0,15.0,20.0,25.5,43.0,81.0
number_diagnoses_mean,71518.0,7.29142,1.917506,1.0,6.0,8.0,9.0,9.0,9.0,16.0
medication_active_entries_mean,71518.0,1.176481,0.904391,0.0,0.666667,1.0,2.0,2.0,4.0,6.0
distinct_medications_recorded_active,71518.0,1.295478,1.00312,0.0,1.0,1.0,2.0,3.0,4.0,7.0


## 11. Manual inspection of several patients

Inspect a deterministic sample of patients and compare their representation with their
underlying encounter records.

This is important because a mathematically valid aggregation can still be clinically
mis-specified.

In [14]:
sample_patients = patient_representation["patient_nbr"].head(5).tolist()

print("Sample patient IDs:", sample_patients)

display(
    patient_representation[
        patient_representation["patient_nbr"].isin(sample_patients)
    ].T
)

print("\nCorresponding encounter counts:")
print(
    df[df["patient_nbr"].isin(sample_patients)]
    .groupby("patient_nbr")
    .size()
    .rename("raw_encounter_count")
)

Sample patient IDs: [135, 378, 729, 774, 927]


,0,1,2,3,4
patient_nbr,135,378,729,774,927
encounter_count,2,1,1,1,1
number_inpatient_sum,1,0,0,0,0
number_inpatient_mean,0.5,0.0,0.0,0.0,0.0
number_inpatient_max,1,0,0,0,0
number_emergency_sum,0,0,0,0,0
number_emergency_mean,0.0,0.0,0.0,0.0,0.0
number_emergency_max,0,0,0,0,0
number_outpatient_sum,0,0,0,0,0
number_outpatient_mean,0.0,0.0,0.0,0.0,0.0



Corresponding encounter counts:
patient_nbr
135    2
378    1
729    1
774    1
927    1
Name: raw_encounter_count, dtype: int64


## 12. Representation dictionary

This file documents what each generated patient-level variable means.

In [15]:
dictionary_rows = [
    {
        "variable": "patient_nbr",
        "group": "identifier",
        "definition": "Patient identifier used as the patient/network node identifier.",
        "used_as_feature": False,
        "notes": "Identifier only."
    },
    {
        "variable": "encounter_count",
        "group": "utilization",
        "definition": "Number of encounter records available for the patient.",
        "used_as_feature": True,
        "notes": "Not a temporal prior-utilization measure."
    },
]

for c in UTILIZATION_COLS:
    for stat in ["sum", "mean", "max"]:
        name = f"{c}_{stat}"
        if name in patient_representation.columns:
            dictionary_rows.append({
                "variable": name,
                "group": "utilization",
                "definition": f"{stat} of {c} across available encounters for the patient.",
                "used_as_feature": True,
                "notes": "Temporal ordering is unavailable."
            })

for c in COUNT_SUMMARY_COLS:
    for stat in ["sum", "mean", "max"]:
        name = f"{c}_{stat}"
        if name in patient_representation.columns:
            dictionary_rows.append({
                "variable": name,
                "group": "clinical_count",
                "definition": f"{stat} of {c} across available encounters.",
                "used_as_feature": True,
                "notes": "Temporal ordering is unavailable."
            })

for c in [
    "medication_active_entries_sum",
    "medication_active_entries_mean",
    "medication_active_entries_max",
    "distinct_medications_recorded_active",
]:
    if c in patient_representation.columns:
        dictionary_rows.append({
            "variable": c,
            "group": "medication_burden",
            "definition": "Patient-level medication-status burden summary.",
            "used_as_feature": True,
            "notes": "Non-missing values other than 'No' are treated as recorded active/change status."
        })

for c in [
    "diagnosis_entries_sum",
    "diagnosis_entries_mean",
    "diagnosis_entries_max",
    "unique_diagnosis_codes",
]:
    if c in patient_representation.columns:
        dictionary_rows.append({
            "variable": c,
            "group": "diagnosis_burden",
            "definition": "Patient-level diagnosis burden summary.",
            "used_as_feature": True,
            "notes": "Diagnosis codes are not outcome-derived."
        })

for c in CONTEXT_MODE_COLS:
    name = f"{c}_mode"
    if name in patient_representation.columns:
        dictionary_rows.append({
            "variable": name,
            "group": "demographic_context",
            "definition": f"Deterministic patient-level mode of {c}.",
            "used_as_feature": True,
            "notes": "Ties resolved lexicographically; no encounter order assumed."
        })

representation_dictionary = pd.DataFrame(dictionary_rows)

# Add excluded variables for traceability
for c in EXCLUDED_FROM_REPRESENTATION:
    if c in df.columns:
        reason = "identifier" if c in IDENTIFIER_COLS else (
            "target" if c == TARGET_COL else (
                "review_required" if c in REVIEW_REQUIRED_COLS else ">=90% missing"
            )
        )
        representation_dictionary = pd.concat([
            representation_dictionary,
            pd.DataFrame([{
                "variable": c,
                "group": "excluded",
                "definition": "Not included in current patient feature representation.",
                "used_as_feature": False,
                "notes": reason
            }])
        ], ignore_index=True)

representation_dictionary

,variable,group,definition,used_as_feature,notes
0,patient_nbr,identifier,Patient identifier used as the patient/network...,False,Identifier only.
1,encounter_count,utilization,Number of encounter records available for the ...,True,Not a temporal prior-utilization measure.
2,number_inpatient_sum,utilization,sum of number_inpatient across available encou...,True,Temporal ordering is unavailable.
3,number_inpatient_mean,utilization,mean of number_inpatient across available enco...,True,Temporal ordering is unavailable.
4,number_inpatient_max,utilization,max of number_inpatient across available encou...,True,Temporal ordering is unavailable.
5,number_emergency_sum,utilization,sum of number_emergency across available encou...,True,Temporal ordering is unavailable.
6,number_emergency_mean,utilization,mean of number_emergency across available enco...,True,Temporal ordering is unavailable.
7,number_emergency_max,utilization,max of number_emergency across available encou...,True,Temporal ordering is unavailable.
8,number_outpatient_sum,utilization,sum of number_outpatient across available enco...,True,Temporal ordering is unavailable.
9,number_outpatient_mean,utilization,mean of number_outpatient across available enc...,True,Temporal ordering is unavailable.


## 13. Save Notebook 03 outputs

Outputs:

- `03_patient_representation.csv` — one row per patient
- `03_target_summary_not_for_features.csv` — separate outcome summary
- `03_representation_dictionary.csv` — variable definitions
- `03_distribution_summary.csv` — distribution diagnostics
- `03_representation_summary.json` — reproducibility/checkpoint summary

In [17]:
PATIENT_OUTPUT = RESULTS_DIR / "03_patient_representation.csv"
TARGET_OUTPUT = RESULTS_DIR / "03_target_summary_not_for_features.csv"
DICT_OUTPUT = RESULTS_DIR / "03_representation_dictionary.csv"
DIST_OUTPUT = RESULTS_DIR / "03_distribution_summary.csv"
SUMMARY_OUTPUT = RESULTS_DIR / "03_representation_summary.json"

# Save the main outputs
patient_representation.to_csv(PATIENT_OUTPUT, index=False)
target_summary.to_csv(TARGET_OUTPUT, index=False)
representation_dictionary.to_csv(DICT_OUTPUT, index=False)
distribution_summary.to_csv(DIST_OUTPUT)

# Convert NumPy/Pandas scalar types into normal Python types
def json_safe(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    return value

summary = {
    "notebook": "03_patient_representation",
    "seed": int(SEED),
    "input_file": str(CLEAN_PATH),
    "input_encounters": int(len(df)),
    "input_unique_patients": int(df["patient_nbr"].nunique()),
    "output_patient_rows": int(len(patient_representation)),
    "output_feature_columns": int(len(patient_representation.columns) - 1),
    "target_rows": int(len(target_summary)),
    "medication_columns_detected": [str(x) for x in MEDICATION_COLS],
    "diagnosis_columns_detected": [str(x) for x in DIAG_COLS],
    "utilization_columns": [str(x) for x in UTILIZATION_COLS],
    "clinical_count_columns": [str(x) for x in COUNT_SUMMARY_COLS],
    "context_columns": [str(x) for x in CONTEXT_MODE_COLS],
    "excluded_columns": [str(x) for x in EXCLUDED_FROM_REPRESENTATION],
    "checks": {
        str(k): bool(v)
        for k, v in checks.items()
    },
    "temporal_limitation": (
        "No encounter timestamps are available in the dataset; cumulative observed "
        "utilization is not labelled as prior utilization. Notebook 04 must define "
        "the leakage-safe prediction/intervention point."
    ),
}

SUMMARY_OUTPUT.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("Saved:")
for p in [
    PATIENT_OUTPUT,
    TARGET_OUTPUT,
    DICT_OUTPUT,
    DIST_OUTPUT,
    SUMMARY_OUTPUT
]:
    print(" -", p)

Saved:
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\03_patient_representation.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\03_target_summary_not_for_features.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\03_representation_dictionary.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\03_distribution_summary.csv
 - c:\Users\Gayatri\OneDrive\Desktop\sna\results\03_representation_summary.json


# 14. FINAL CHECKPOINT — Notebook 03

### GO criteria from the project proposal

- Number of patient rows equals `nunique(patient_nbr)`
- No patient has a duplicated final representation
- Patient-level representation exists
- Readmission target is clearly separated
- Identifier/target leakage is absent from the feature table
- Core utilization, medication, diagnosis, procedure, and demographic/context groups are represented
- Outputs are saved

### STOP if any check fails.

Do not proceed to Notebook 04 until this checkpoint is PASS and the sample patients /
distributions have been manually reviewed.

In [18]:
# Core representation-group checks
required_groups = {
    "encounter_count": "encounter_count" in patient_representation.columns,
    "inpatient_utilization": any(c.startswith("number_inpatient_") for c in patient_representation.columns),
    "emergency_utilization": any(c.startswith("number_emergency_") for c in patient_representation.columns),
    "outpatient_utilization": any(c.startswith("number_outpatient_") for c in patient_representation.columns),
    "medication_burden": any("medication" in c.lower() for c in patient_representation.columns),
    "diagnosis_burden": any("diagnosis" in c.lower() for c in patient_representation.columns),
    "procedure_burden": any("num_procedures_" in c.lower() for c in patient_representation.columns),
    "demographic_context": len(CONTEXT_MODE_COLS) > 0,
}

final_checks = {
    "patient_row_count_equals_nunique": len(patient_representation) == df["patient_nbr"].nunique(),
    "no_duplicate_patient_representation": not patient_representation["patient_nbr"].duplicated().any(),
    "patient_id_present": "patient_nbr" in patient_representation.columns,
    "encounter_id_not_in_features": "encounter_id" not in patient_representation.columns,
    "readmission_not_in_features": "readmitted" not in patient_representation.columns,
    "target_preserved_separately": len(target_summary) == len(patient_representation),
    "numeric_values_finite": finite_ok,
    "count_variables_nonnegative": nonnegative_ok,
    "patient_output_exists": PATIENT_OUTPUT.exists(),
    "target_output_exists": TARGET_OUTPUT.exists(),
    "dictionary_output_exists": DICT_OUTPUT.exists(),
    "distribution_output_exists": DIST_OUTPUT.exists(),
    **required_groups,
}

print("=" * 75)
print("NOTEBOOK 03 — FINAL CHECKPOINT")
print("=" * 75)

for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL':<6} | {name}")

print("=" * 75)

if all(final_checks.values()):
    print("OVERALL RESULT: PASS")
    print("Proceed to MANUAL REVIEW.")
    print("If manual review is satisfactory, proceed to Notebook 04.")
else:
    print("OVERALL RESULT: FAIL")
    print("Fix the failed checks before moving to Notebook 04.")

NOTEBOOK 03 — FINAL CHECKPOINT
PASS   | patient_row_count_equals_nunique
PASS   | no_duplicate_patient_representation
PASS   | patient_id_present
PASS   | encounter_id_not_in_features
PASS   | readmission_not_in_features
PASS   | target_preserved_separately
PASS   | numeric_values_finite
PASS   | count_variables_nonnegative
PASS   | patient_output_exists
PASS   | target_output_exists
PASS   | dictionary_output_exists
PASS   | distribution_output_exists
PASS   | encounter_count
PASS   | inpatient_utilization
PASS   | emergency_utilization
PASS   | outpatient_utilization
PASS   | medication_burden
PASS   | diagnosis_burden
PASS   | procedure_burden
PASS   | demographic_context
OVERALL RESULT: PASS
Proceed to MANUAL REVIEW.
If manual review is satisfactory, proceed to Notebook 04.
